# Ropedia Academy — A3 · Temporal motion backbones

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A3.ipynb)

> **Encodes a motion sequence with a temporal backbone and plots joint position vs translation-invariant velocity over time.**
>
> 用时序骨干编码运动序列，并画出关节位置与平移不变的速度随时间的曲线。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A3

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt

T, J = 64, 17                                   # 64 frames, 17 joints
t = torch.linspace(0, 4*3.14159, T)
seq = torch.stack([torch.sin(t + j) for j in range(J)], 1)[None, :, :, None] * torch.ones(1,T,J,3)

# Representation matters: VELOCITIES are translation-invariant (same action, anywhere).
shifted = seq + torch.tensor([5., 0., 0.])
vel, vel2 = seq[:,1:]-seq[:,:-1], shifted[:,1:]-shifted[:,:-1]
print("velocity is translation-invariant:", torch.allclose(vel, vel2, atol=1e-5))

# A temporal backbone (1D conv over time) encodes MOVEMENT, not a single pose.
x = seq.reshape(1, T, J*3).transpose(1, 2)
tcn = nn.Sequential(nn.Conv1d(J*3, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
print("temporal-conv motion feature:", tuple(tcn(x).squeeze(-1).shape))

# Visualize: a joint's position vs its velocity over time.
plt.figure(figsize=(6, 3))
plt.plot(seq[0,:,5,0], label="joint position x")
plt.plot(range(1,T), vel[0,:,5,0], label="velocity (Δ)")
plt.title("motion is a sequence; velocity exposes the dynamics")
plt.xlabel("frame"); plt.legend(); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A3
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks